# Document Q&A capstone — public demo

This notebook adapts lessons from the document-AI work I did during my Pfizer externship: extract digital PDF text, use OCR for scanned pages, retrieve relevant passages, and show their page sources. Start with the fictional public sample at `data/public-fixtures/synthetic-quality-documents.pdf` from the candidate root. If you launch the notebook with `notebooks/` as the working directory, the same file is `../data/public-fixtures/synthetic-quality-documents.pdf`.

The original LlamaIndex and Gradio pipeline needs a compatible GPU/runtime, Tesseract and Poppler for scanned pages, and downloads the configured embedding and Mistral model weights when its setup cell runs. Those model cells were not run for this local offline check. Gradio launches on localhost by default; change sharing only intentionally.


In [ ]:
# Install the original pipeline dependencies in your chosen runtime; OCR uses Tesseract and Poppler.
!apt-get -qq install -y tesseract-ocr poppler-utils
!pip install -q gradio PyPDF2 llama-index llama-index-embeddings-huggingface llama-cpp-python llama-index-llms-llama-cpp langchain-text-splitters pytesseract pdf2image


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = LlamaCPP(
    model_url="https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf",
    temperature=0.1, max_new_tokens=256, context_window=3900,
    model_kwargs={"n_gpu_layers": -1}, verbose=False,
)

In [ ]:
import gradio as gr
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from llama_index.core import Document, VectorStoreIndex

INDEX = {"idx": None, "info": ""}   # global so handle_chat can reach it

def extract_page_text(reader, i, path):
    """Digital text first; fall back to OCR for scanned/image pages."""
    txt = reader.pages[i].extract_text() or ""
    if txt.strip():
        return txt
    # scanned page → OCR
    from pdf2image import convert_from_path
    import pytesseract
    img = convert_from_path(path, first_page=i + 1, last_page=i + 1)[0]
    return pytesseract.image_to_string(img)

def process_pdf(file):
    if file is None:
        return "⚠️ Please upload a PDF first."
    reader = PdfReader(file.name)
    fname = file.name.split("/")[-1]

    splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100)
    docs = []
    ci = 0
    for pnum in range(len(reader.pages)):
        ptext = extract_page_text(reader, pnum, file.name)
        for chunk in splitter.split_text(ptext):
            docs.append(Document(text=chunk, metadata={
                "doc_type": "Fictional Quality Document",
                "chunk_index": ci,
                "page_start": pnum, "page_end": pnum,   # ⭐ page range
                "source_file": fname,
            }))
            ci += 1

    INDEX["idx"] = VectorStoreIndex.from_documents(docs)
    INDEX["info"] = f"✅ Processed '{fname}' — {len(reader.pages)} pages, {len(docs)} chunks indexed."
    return INDEX["info"]


In [ ]:
def handle_chat(message, history):
    if INDEX["idx"] is None:
        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": "⚠️ Please upload and process a PDF first."},
        ]
        return history, ""

    retriever = INDEX["idx"].as_retriever(similarity_top_k=4)
    nodes = retriever.retrieve(message)

    if not nodes:
        answer = "I couldn't find relevant information to answer your question."
    else:
        context = "\n\n".join(
            f"[Source: {n.metadata.get('source_file')}, page {n.metadata.get('page_start')}]\n{n.text}"
            for n in nodes
        )
        prompt = (
            "You are a helpful assistant answering questions about a fictional quality document.\n"
            "Use ONLY the context below. If the answer isn't there, say so. Cite sources.\n\n"
            f"Context:\n{context}\n\nQuestion: {message}\n\nAnswer:"
        )
        answer = Settings.llm.complete(prompt).text.strip()

        avg_conf = sum(n.score for n in nodes) / len(nodes)
        pages = sorted({n.metadata.get("page_start") for n in nodes})
        srcs = ", ".join(sorted({n.metadata.get("source_file", "?") for n in nodes}))
        answer += (f"\n\n---\n📄 Sources: {srcs} (pages {pages}) · "
                   f"🎯 Confidence: {avg_conf:.2f} · 🧩 Chunks used: {len(nodes)}")

    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    return history, ""


In [ ]:
# Upload ../data/public-fixtures/synthetic-quality-documents.pdf as the first example.
with gr.Blocks(title="Fictional Document Q&A Chatbot") as demo:
    gr.Markdown("## 📄 Fictional Document Q&A Chatbot\nUpload the fictional public PDF and ask questions — answers cite sources and show confidence.")
    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
            process_btn = gr.Button("🔄 Process Document", variant="primary")
            status = gr.Textbox(label="Document Info", interactive=False)
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat History", height=450)
            user_input = gr.Textbox(placeholder="Ask a question about your document...", label="Your Question")
            with gr.Row():
                send_btn = gr.Button("📤 Send", variant="primary")
                clear_btn = gr.Button("🗑️ Clear Chat")

    process_btn.click(process_pdf, inputs=pdf_input, outputs=status)
    send_btn.click(handle_chat, inputs=[user_input, chatbot], outputs=[chatbot, user_input])
    user_input.submit(handle_chat, inputs=[user_input, chatbot], outputs=[chatbot, user_input])
    clear_btn.click(lambda: [], outputs=chatbot)

demo.launch(share=False, theme=gr.themes.Soft())
